In [ ]:
#Use this script to generate train/dev/test splits

In [1]:
import warnings
warnings.filterwarnings("ignore", message="unknown class")

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# Load the formatted data
df = pd.read_csv(
    "pilot_split.csv",
    dtype=str,                 # keep grams as strings
    keep_default_na=False,     # don't auto-treat "null"/"NA" as NaN
    na_values=[]               # no extra NA markers
)

# Features (X) and labels (y)
X = df[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol", "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]]  # adjust as needed; taken out for now: , "ant_complete", "ant_initial", "ant_none"
y = df[["eng_gram", "deu_gram", "cmn_gram"]] #taken out for now: , "kat_gram", "jpn_gram"

# Filter label classes that occur fewer than twice
y = df[["eng_gram", "deu_gram", "cmn_gram"]] #taken out for now: , "kat_gram", "jpn_gram"
X = df[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol",
        "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]] #taken out for now: , "ant_complete", "ant_initial", "ant_none"

label_counts = y.value_counts()
valid_labels = label_counts[label_counts >= 2].index

# Define which gram columns to transform
gram_cols = ["eng_gram", "deu_gram", "cmn_gram"]

# Convert space-separated strings into lists — apply directly to y
for col in gram_cols:
    y[col] = y[col].apply(lambda x: [] if x == "n_a" else x.split())

# Quick check
print(y["deu_gram"].head(10).tolist())
print(type(y["deu_gram"].iloc[0]))

[['prat', 'perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['perfekt'], [], []]
<class 'list'>
eng_gram: 6 labels → ['might' 'past' 'perfect' 'prog' 'should' 'would']...
deu_gram: 5 labels → ['k1' 'k2' 'perfekt' 'prasens' 'prat']...
cmn_gram: 10 labels → ['guo' 'le' 'meiyou' 'nullmarked' 'qilai' 'suo_de' 'yi' 'yijing' 'zaojiu'
 'zaoyi']...


/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_47464/2990643299.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[col] = y[col].apply(lambda x: [] if x == "n_a" else x.split())


In [ ]:
# Proceed with splitting -- 50/50
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.5, random_state=42)

# Proceed with splitting -- 50/25/25
#X_temp, X_test, y_temp, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42)
#X_train, X_dev, y_train, y_dev = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Remove n_a rows separately per language
X_train_no_na = {}
X_dev_no_na = {}
y_train_no_na = {}
y_dev_no_na = {}

#Define paradigms

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    mask_train = ~y_train[lang].apply(lambda lbls: lbls == [] or lbls == ['n_a'])
    mask_dev   = ~y_dev[lang].apply(lambda lbls: lbls == [] or lbls == ['n_a'])

    #For paradigm in paradigms...
    
    # Keep separate copies of filtered data for this language
    X_train_no_na[lang] = X_train.loc[mask_train].reset_index(drop=True)
    X_dev_no_na[lang]   = X_dev.loc[mask_dev].reset_index(drop=True)
    y_train_no_na[lang] = y_train.loc[mask_train, lang].reset_index(drop=True)
    y_dev_no_na[lang]   = y_dev.loc[mask_dev, lang].reset_index(drop=True)

In [ ]:
# Remove rows where any language gram is n_a before training
#mask_no_na = ~y_train.applymap(lambda lbls: lbls == [] or lbls == ['n_a']).any(axis=1)
#X_train = X_train[mask_no_na].reset_index(drop=True)
#y_train = y_train[mask_no_na].reset_index(drop=True)

#mask_no_na_dev = ~y_dev.applymap(lambda lbls: lbls == [] or lbls == ['n_a']).any(axis=1)
#X_dev = X_dev[mask_no_na_dev].reset_index(drop=True)
#y_dev = y_dev[mask_no_na_dev].reset_index(drop=True)

X_train_model = X_train.drop(columns=["TokenID"])
X_dev_model   = X_dev.drop(columns=["TokenID"])

mlbs = {}
y_train_bin = {}
y_dev_bin   = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    mlb = MultiLabelBinarizer()
    y_train_bin[lang] = mlb.fit_transform(y_train_no_na[lang])  # <- use *_no_na
    y_dev_bin[lang]   = mlb.transform(y_dev_no_na[lang])        # <- use *_no_na ...no_na[lang, pname]
    mlbs[lang] = mlb
    print(f"{lang}: {len(mlb.classes_)} labels → {mlb.classes_[:10]}...")

In [2]:
print(len(df))

301


In [3]:
print(y["deu_gram"].head(10).tolist())
print(type(y["deu_gram"].iloc[0]))

[['prat', 'perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['perfekt'], [], []]
<class 'list'>


In [ ]:
#For train/dev/test
#print(f"Train: {len(X_train)} rows")
#print(f"Dev: {len(X_dev)} rows")
#print(f"Test: {len(X_test)} rows")

#For train/dev only
#print(f"Train: {len(X_train)} rows")
#print(f"Dev: {len(X_dev)} rows")

#print("Train label distribution:")
#print(y_train.value_counts(normalize=True))
#print("\nDev label distribution:")
#print(y_dev.value_counts(normalize=True))
#Include the following for train/dev/test split
#print("\nTest label distribution:")
#print(y_test.value_counts(normalize=True))

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train and predict for each language
for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    print(f"\n=== Classification Report for {lang.upper()} ===")

    # Initialize a new classifier for this language
    clf = RandomForestClassifier(random_state=42)

    # Train only on non-n_a rows for this language
    X_train_lang = X_train_no_na[lang].drop(columns=["TokenID"])
    X_dev_lang   = X_dev_no_na[lang].drop(columns=["TokenID"])

    clf.fit(X_train_lang, y_train_bin[lang])
    preds = clf.predict(X_dev_lang)

    # Inverse transform predictions back into readable labels
    pred_sets = mlbs[lang].inverse_transform(preds)
    true_sets = y_dev_no_na[lang]

    # Keep alignment by rebuilding together
    pairs = [(t, p) for t, p in zip(true_sets, pred_sets) if t and p]

    true_vals = pd.Series([", ".join(t) for t, _ in pairs])
    pred_vals = pd.Series([", ".join(p) for _, p in pairs])

    # Exclude n_a rows for reporting only
    mask = true_vals != "n_a"
    true_vals = true_vals[mask]
    pred_vals = pred_vals[mask]

    # Summary + sanity checks
    print("Shapes → true_vals:", true_vals.shape, "pred_vals:", pred_vals.shape)
    print("First 10 true:", true_vals.head(10).tolist())
    print("First 10 pred:", pred_vals.head(10).tolist())
    print("Manual accuracy:", (true_vals == pred_vals).mean())

    # sklearn report
    print(classification_report(true_vals, pred_vals, zero_division=0))


=== Classification Report for ENG_GRAM ===
Shapes → true_vals: (151,) pred_vals: (151,)
First 10 true: ['perfect', 'perfect', 'perfect', 'past, perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect']
First 10 pred: ['perfect', 'perfect', 'perfect', 'past, perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect']
Manual accuracy: 0.8940397350993378
                 precision    recall  f1-score   support

 could, perfect       0.00      0.00      0.00         3
  past, perfect       0.76      0.81      0.79        32
        perfect       0.95      0.99      0.97       110
 perfect, would       0.00      0.00      0.00         0
should, perfect       0.00      0.00      0.00         2
 would, perfect       0.00      0.00      0.00         4

       accuracy                           0.89       151
      macro avg       0.29      0.30      0.29       151
   weighted avg       0.85      0.89      0.87       151


=== Classification Report for DEU_GRA

In [ ]:
from sklearn.metrics import confusion_matrix

for i, lang in enumerate(y_dev.columns):
    print(f"\n=== Confusion Matrix for {lang} ===")
    true_vals = y_dev[lang]
    pred_vals = rfc_pred[:, i]   # <-- was y_pred
    print(confusion_matrix(true_vals, pred_vals))

In [ ]:
# set output folder once
OUTPUT_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Outputs"

from sklearn.metrics import classification_report
report_dfs = {}

for i, lang in enumerate(y_dev.columns):
    true_vals = y_dev[lang]
    pred_vals = rfc_pred[:, i]   # <-- was y_pred
    report_dict = classification_report(true_vals, pred_vals, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_dfs[lang] = report_df
    report_df.to_csv(f"{OUTPUT_DIR}/{lang}_rfc_classification_report.csv")

print("Classification reports exported to:", OUTPUT_DIR)

In [ ]:
from sklearn.metrics import confusion_matrix

for i, lang in enumerate(y_dev.columns):
    true_vals = y_dev[lang]
    pred_vals = rfc_pred[:, i]   # <-- was y_pred
    cm = confusion_matrix(true_vals, pred_vals)
    cm_df = pd.DataFrame(cm, 
                         index=[f"True_{label}" for label in sorted(true_vals.unique())], 
                         columns=[f"Pred_{label}" for label in sorted(true_vals.unique())])
    cm_df.to_csv(f"{OUTPUT_DIR}/{lang}_rfc_confusion_matrix.csv")

print("Confusion matrices exported.")

In [ ]:
# Export features and predictions [features once; preds per experiment]
X_dev.to_csv(f"{OUTPUT_DIR}/X_dev.csv")
y_dev.to_csv(f"{OUTPUT_DIR}/y_dev.csv")
pd.DataFrame(rfc_pred, columns=y_dev.columns).to_csv(f"{OUTPUT_DIR}/y_pred_rfc.csv")

In [ ]:
# Build a single aligned table with features, golds, and RFC predictions
dev_rfc = X_dev.copy()

# gold labels per language
for lang in y_dev.columns:
    dev_rfc[f'gold_{lang}'] = y_dev[lang]

# RFC predictions per language
for i, lang in enumerate(y_dev.columns):
    dev_rfc[f'pred_rfc_{lang}'] = rfc_pred[:, i]

# Optional error flags (handy for F/A counts later)
for lang in y_dev.columns:
    dev_rfc[f'err_rfc_{lang}'] = (dev_rfc[f'gold_{lang}'] != dev_rfc[f'pred_rfc_{lang}']).astype(int)

# Sanity checks
assert rfc_pred.shape == (len(X_dev), len(y_dev.columns))
assert dev_rfc.notna().all().all()

# Save
dev_rfc.to_csv(f"{OUTPUT_DIR}/dev_with_rfc.csv", index=False)
print("Saved:", f"{OUTPUT_DIR}/dev_with_rfc.csv")

In [ ]:
print("Rows:", len(dev_rfc))
for lang in y_dev.columns:
    N = (dev_rfc[f'gold_{lang}'] != 'n_a').sum()
    E = dev_rfc.loc[dev_rfc[f'gold_{lang}'] != 'n_a', f'err_rfc_{lang}'].sum()
    print(f"{lang}: N={N}, E={E}")